# 04 Read Results (Part 2 / RegGS)

Goal:

1. scan the current `output/part2/` tree directly
2. detect complete runs by `eval_train.json` + any `eval_test*.json` + `ate_aligned.json`
3. parse run naming fields (dataset/scene/ckpt/sr/nv/sm/profile) with backward compatibility
4. collect run-level metrics and frame-level metrics for multiple test protocols
5. export clean csv summaries into `results/part2/`

Notes:
- Part 2 has no convergence logs in this protocol.
- Test protocols can differ (`sampled_test`, `all_non_train`, or tagged outputs).
- ATE values are kept, but cross-dataset ATE comparison is marked as not recommended.

In [ ]:
from pathlib import Path
import json
import re
import numpy as np
import pandas as pd

CV_ROOT = Path('~/CV_Project').expanduser()
OUTPUT_ROOT = CV_ROOT / 'output' / 'part2'
RESULTS_ROOT = CV_ROOT / 'results' / 'part2' / 'part2_reggs'

FINAL_DIR = RESULTS_ROOT / 'final'
FRAME_DIR = RESULTS_ROOT / 'frame'
QC_DIR = RESULTS_ROOT / 'qc'

for d in [FINAL_DIR, FRAME_DIR, QC_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('OUTPUT_ROOT =', OUTPUT_ROOT)
print('RESULTS_ROOT =', RESULTS_ROOT)
print('FINAL_DIR =', FINAL_DIR)
print('FRAME_DIR =', FRAME_DIR)
print('QC_DIR =', QC_DIR)

OUTPUT_ROOT = /home/bzhang512/CV_Project/output/part2
RESULTS_ROOT = /home/bzhang512/CV_Project/results/part2
FINAL_DIR = /home/bzhang512/CV_Project/results/part2/final
FRAME_DIR = /home/bzhang512/CV_Project/results/part2/frame
QC_DIR = /home/bzhang512/CV_Project/results/part2/qc


## 1. Scan run directories and build run inventory

In [2]:
required_core_files = ['eval_train.json', 'ate_aligned.json']

DATASET_NAME_ALIASES = {
    're10k': 're10k_1',
}

def normalize_dataset_name(name: str) -> str:
    return DATASET_NAME_ALIASES.get(str(name), str(name))

def parse_run_name_fields(dataset: str, run_name: str):
    out = {
        'scene_id': 'main',
        'ckpt_from_name': np.nan,
        'sample_rate_from_name': np.nan,
        'n_views_from_name': np.nan,
        'submap_every_from_name': np.nan,
        'profile_from_name': np.nan,
        'parse_pattern': 'fallback',
    }

    if dataset == '405841':
        m_new = re.match(
            r'^reggs_405841_scene_([A-Za-z0-9]+)_([A-Za-z0-9]+)-ckpt_sr(\d+)_nv(\d+)_sm(\d+)_(.+)$',
            run_name,
        )
        if m_new:
            out.update({
                'scene_id': m_new.group(1),
                'ckpt_from_name': m_new.group(2),
                'sample_rate_from_name': int(m_new.group(3)),
                'n_views_from_name': int(m_new.group(4)),
                'submap_every_from_name': int(m_new.group(5)),
                'profile_from_name': m_new.group(6),
                'parse_pattern': '405841_new',
            })
            return out

        m_old = re.match(r'^reggs_405841_scene_([A-Za-z0-9]+)_sr(\d+)_nv(\d+)$', run_name)
        if m_old:
            out.update({
                'scene_id': m_old.group(1),
                'sample_rate_from_name': int(m_old.group(2)),
                'n_views_from_name': int(m_old.group(3)),
                'parse_pattern': '405841_old',
            })
            return out

    m_general_new = re.match(
        r'^reggs_([A-Za-z0-9]+)_([A-Za-z0-9]+)-ckpt_sr(\d+)_nv(\d+)_sm(\d+)_(.+)$',
        run_name,
    )
    if m_general_new:
        out.update({
            'ckpt_from_name': m_general_new.group(2),
            'sample_rate_from_name': int(m_general_new.group(3)),
            'n_views_from_name': int(m_general_new.group(4)),
            'submap_every_from_name': int(m_general_new.group(5)),
            'profile_from_name': m_general_new.group(6),
            'parse_pattern': 'general_new',
        })
        return out

    m_general_old = re.match(r'^reggs_([A-Za-z0-9]+)_sr(\d+)_nv(\d+)$', run_name)
    if m_general_old:
        out.update({
            'sample_rate_from_name': int(m_general_old.group(2)),
            'n_views_from_name': int(m_general_old.group(3)),
            'parse_pattern': 'general_old',
        })
        return out

    sr_match = re.search(r'_sr(\d+)', run_name)
    nv_match = re.search(r'_nv(\d+)', run_name)
    sm_match = re.search(r'_sm(\d+)', run_name)
    ckpt_match = re.search(r'_([A-Za-z0-9]+)-ckpt_', run_name)
    scene_match = re.search(r'scene_([A-Za-z0-9]+)', run_name)
    profile_match = re.search(r'_sm\d+_(.+)$', run_name)

    if sr_match:
        out['sample_rate_from_name'] = int(sr_match.group(1))
    if nv_match:
        out['n_views_from_name'] = int(nv_match.group(1))
    if sm_match:
        out['submap_every_from_name'] = int(sm_match.group(1))
    if ckpt_match:
        out['ckpt_from_name'] = ckpt_match.group(1)
    if scene_match:
        out['scene_id'] = scene_match.group(1)
    if profile_match:
        out['profile_from_name'] = profile_match.group(1)

    return out

dataset_dirs = sorted([d for d in OUTPUT_ROOT.iterdir() if d.is_dir()]) if OUTPUT_ROOT.exists() else []
candidate_run_dirs = []
for dataset_dir in dataset_dirs:
    run_dirs = sorted([d for d in dataset_dir.iterdir() if d.is_dir()])
    candidate_run_dirs.extend(run_dirs)

inventory_rows = []
for run_dir in candidate_run_dirs:
    dataset_raw = run_dir.parent.name
    dataset = normalize_dataset_name(dataset_raw)
    run_name = run_dir.name
    rel_run_dir = run_dir.relative_to(OUTPUT_ROOT)

    core_exists = {f"exists_{name.replace('.json', '')}": (run_dir / name).exists() for name in required_core_files}
    test_metric_files = sorted(run_dir.glob('eval_test*.json'))
    exists_eval_test_any = len(test_metric_files) > 0

    missing_files = [name for name in required_core_files if not (run_dir / name).exists()]
    if not exists_eval_test_any:
        missing_files.append('eval_test*.json')

    is_complete = len(missing_files) == 0
    parsed = parse_run_name_fields(dataset, run_name)
    run_mtime = run_dir.stat().st_mtime if run_dir.exists() else np.nan

    inventory_rows.append({
        'dataset': dataset,
        'run_name': run_name,
        'experiment_name': f'{dataset}__{run_name}',
        'run_dir': str(run_dir),
        'run_dir_rel': str(rel_run_dir),
        **parsed,
        'run_mtime': run_mtime,
        **core_exists,
        'exists_eval_test_any': exists_eval_test_any,
        'n_test_metric_files': len(test_metric_files),
        'test_metric_files': ';'.join(p.name for p in test_metric_files),
        'missing_files': ';'.join(missing_files),
        'is_complete': is_complete,
        'skip_reason': '' if is_complete else f"missing:{';'.join(missing_files)}",
    })

inventory_df = pd.DataFrame(inventory_rows)
if inventory_df.empty:
    inventory_df = pd.DataFrame(columns=[
        'dataset', 'run_name', 'experiment_name', 'run_dir', 'run_dir_rel',
        'scene_id', 'ckpt_from_name',
        'sample_rate_from_name', 'n_views_from_name', 'submap_every_from_name', 'profile_from_name', 'parse_pattern',
        'run_mtime',
        'exists_eval_train', 'exists_ate_aligned', 'exists_eval_test_any',
        'n_test_metric_files', 'test_metric_files',
        'missing_files', 'is_complete', 'skip_reason'
    ])

complete_runs_df = inventory_df[inventory_df['is_complete']].copy()
skipped_runs_df = inventory_df[~inventory_df['is_complete']].copy()

print('num candidate runs =', len(inventory_df))
print('num complete runs =', len(complete_runs_df))
print('num skipped runs =', len(skipped_runs_df))
inventory_df.sort_values(['dataset', 'run_name']).head(20)

num candidate runs = 10
num complete runs = 9
num skipped runs = 1


,dataset,run_name,experiment_name,run_dir,run_dir_rel,scene_id,ckpt_from_name,sample_rate_from_name,n_views_from_name,submap_every_from_name,...,parse_pattern,run_mtime,exists_eval_train,exists_ate_aligned,exists_eval_test_any,n_test_metric_files,test_metric_files,missing_files,is_complete,skip_reason
0,405841,reggs_405841_scene_A_sr10_nv4,405841__reggs_405841_scene_A_sr10_nv4,/home/bzhang512/CV_Project/output/part2/405841...,405841/reggs_405841_scene_A_sr10_nv4,A,NaN,10,4,NaN,...,405841_old,1.774771e+09,True,True,True,1,eval_test.json,,True,
1,405841,reggs_405841_scene_C_dl3dv-ckpt_sr10_nv12,405841__reggs_405841_scene_C_dl3dv-ckpt_sr10_nv12,/home/bzhang512/CV_Project/output/part2/405841...,405841/reggs_405841_scene_C_dl3dv-ckpt_sr10_nv12,C,dl3dv,10,12,NaN,...,fallback,1.774794e+09,True,True,True,1,eval_test.json,,True,
2,405841,reggs_405841_scene_C_dl3dv-ckpt_sr30_nv12_sm2_...,405841__reggs_405841_scene_C_dl3dv-ckpt_sr30_n...,/home/bzhang512/CV_Project/output/part2/405841...,405841/reggs_405841_scene_C_dl3dv-ckpt_sr30_nv...,C,dl3dv,30,12,2.0,...,405841_new,1.774808e+09,True,True,True,1,eval_test.json,,True,
3,405841,reggs_405841_scene_full_dl3dv-ckpt_sr30_nv20_s...,405841__reggs_405841_scene_full_dl3dv-ckpt_sr3...,/home/bzhang512/CV_Project/output/part2/405841...,405841/reggs_405841_scene_full_dl3dv-ckpt_sr30...,full,dl3dv,30,20,2.0,...,405841_new,1.774991e+09,False,False,False,0,,eval_train.json;ate_aligned.json;eval_test*.json,False,missing:eval_train.json;ate_aligned.json;eval_...
4,dl3dv_2,reggs_dl3dv2_dl3dv-ckpt_sr30_nv10,dl3dv_2__reggs_dl3dv2_dl3dv-ckpt_sr30_nv10,/home/bzhang512/CV_Project/output/part2/dl3dv_...,dl3dv_2/reggs_dl3dv2_dl3dv-ckpt_sr30_nv10,main,dl3dv,30,10,NaN,...,fallback,1.774789e+09,True,True,True,1,eval_test.json,,True,
5,dl3dv_2,reggs_dl3dv2_dl3dv-ckpt_sr50_nv10_sm2_stable_v1,dl3dv_2__reggs_dl3dv2_dl3dv-ckpt_sr50_nv10_sm2...,/home/bzhang512/CV_Project/output/part2/dl3dv_...,dl3dv_2/reggs_dl3dv2_dl3dv-ckpt_sr50_nv10_sm2_...,main,dl3dv,50,10,2.0,...,general_new,1.774805e+09,True,True,True,1,eval_test.json,,True,
6,dl3dv_2,reggs_dl3dv2_dl3dv-ckpt_sr50_nv10_sm3_stable_v...,dl3dv_2__reggs_dl3dv2_dl3dv-ckpt_sr50_nv10_sm3...,/home/bzhang512/CV_Project/output/part2/dl3dv_...,dl3dv_2/reggs_dl3dv2_dl3dv-ckpt_sr50_nv10_sm3_...,main,dl3dv,50,10,3.0,...,general_new,1.774861e+09,True,True,True,1,eval_test.json,,True,
7,dl3dv_2,reggs_dl3dv2_re10k-ckpt_sr30_nv8,dl3dv_2__reggs_dl3dv2_re10k-ckpt_sr30_nv8,/home/bzhang512/CV_Project/output/part2/dl3dv_...,dl3dv_2/reggs_dl3dv2_re10k-ckpt_sr30_nv8,main,re10k,30,8,NaN,...,fallback,1.774651e+09,True,True,True,1,eval_test.json,,True,
8,re10k_1,reggs_re10k1_re10k-ckpt_sr50_nv9_sm2_compariso...,re10k_1__reggs_re10k1_re10k-ckpt_sr50_nv9_sm2_...,/home/bzhang512/CV_Project/output/part2/re10k_...,re10k_1/reggs_re10k1_re10k-ckpt_sr50_nv9_sm2_c...,main,re10k,50,9,2.0,...,general_new,1.774814e+09,True,True,True,1,eval_test.json,,True,
9,re10k_1,reggs_re10k1_sr30_nv8,re10k_1__reggs_re10k1_sr30_nv8,/home/bzhang512/CV_Project/output/part2/re10k_...,re10k_1/reggs_re10k1_sr30_nv8,main,NaN,30,8,NaN,...,general_old,1.774601e+09,True,True,True,1,eval_test.json,,True,


## 2. Parse run-level and frame-level metrics from the three json files

In [3]:
def read_json_safe(path: Path):
    if not path.exists():
        return None, f'missing:{path.name}'
    try:
        payload = json.loads(path.read_text(encoding='utf-8'))
        if not isinstance(payload, dict):
            return None, f'invalid_root:{path.name}'
        return payload, 'ok'
    except Exception as exc:
        return None, f'json_error:{path.name}:{exc.__class__.__name__}'

def as_float(value):
    try:
        if value is None:
            return np.nan
        return float(value)
    except Exception:
        return np.nan

def ensure_metric_list(payload):
    if not isinstance(payload, dict):
        return []
    metrics = payload.get('metrics', [])
    return metrics if isinstance(metrics, list) else []

def parse_test_metrics_filename(file_name: str):
    if file_name == 'eval_test.json':
        return {
            'test_metrics_file': file_name,
            'test_output_label': 'sampled',
            'test_protocol': 'sampled_test',
            'use_legacy_test_output': True,
        }

    m = re.match(r'^eval_test_(.+)\.json$', file_name)
    if not m:
        return {
            'test_metrics_file': file_name,
            'test_output_label': 'unknown',
            'test_protocol': 'unknown',
            'use_legacy_test_output': False,
        }

    label = m.group(1)
    if label == 'all_non_train':
        protocol = 'all_non_train'
    elif label == 'sampled':
        protocol = 'sampled_test'
    else:
        protocol = 'tagged'

    return {
        'test_metrics_file': file_name,
        'test_output_label': label,
        'test_protocol': protocol,
        'use_legacy_test_output': False,
    }

def test_variant_rank(protocol: str, label: str):
    if label == 'all_non_train' or protocol == 'all_non_train':
        return 0
    if label == 'sampled' or protocol == 'sampled_test':
        return 1
    if protocol == 'tagged':
        return 2
    return 3

run_rows = []
frame_rows = []

for _, inv in complete_runs_df.iterrows():
    run_dir = Path(inv['run_dir'])
    dataset = inv['dataset']
    run_name = inv['run_name']
    experiment_name = inv['experiment_name']

    train_payload, train_status = read_json_safe(run_dir / 'eval_train.json')
    ate_payload, ate_status = read_json_safe(run_dir / 'ate_aligned.json')
    test_metric_files = sorted(run_dir.glob('eval_test*.json'))

    train_metrics = ensure_metric_list(train_payload)

    base_identity = {
        'dataset': dataset,
        'run_name': run_name,
        'experiment_name': experiment_name,
        'run_dir': str(run_dir),
        'run_mtime': inv.get('run_mtime', np.nan),
        'scene_id': inv.get('scene_id', 'main'),
        'ckpt_from_name': inv.get('ckpt_from_name', np.nan),
        'sample_rate_from_name': inv.get('sample_rate_from_name', np.nan),
        'n_views_from_name': inv.get('n_views_from_name', np.nan),
        'submap_every_from_name': inv.get('submap_every_from_name', np.nan),
        'profile_from_name': inv.get('profile_from_name', np.nan),
        'parse_pattern': inv.get('parse_pattern', 'fallback'),
    }

    for test_file in test_metric_files:
        test_payload, test_status = read_json_safe(test_file)
        test_metrics = ensure_metric_list(test_payload)
        test_meta = parse_test_metrics_filename(test_file.name)
        variant_rank = test_variant_rank(test_meta['test_protocol'], test_meta['test_output_label'])

        run_key = f"{dataset}__{run_name}"
        analysis_key = f"{run_key}__test-{test_meta['test_output_label']}"

        run_rows.append({
            **base_identity,
            **test_meta,
            'run_key': run_key,
            'analysis_key': analysis_key,
            'train_json_status': train_status,
            'test_json_status': test_status,
            'ate_json_status': ate_status,
            'train_avg_lpips': as_float((train_payload or {}).get('avg_lpips')),
            'train_avg_psnr': as_float((train_payload or {}).get('avg_psnr')),
            'train_avg_ssim': as_float((train_payload or {}).get('avg_ssim')),
            'test_avg_lpips': as_float((test_payload or {}).get('avg_lpips')),
            'test_avg_psnr': as_float((test_payload or {}).get('avg_psnr')),
            'test_avg_ssim': as_float((test_payload or {}).get('avg_ssim')),
            'n_train_frames': len(train_metrics),
            'n_test_frames': len(test_metrics),
            'test_variant_rank': variant_rank,
            'ate_compared_pose_pairs': as_float((ate_payload or {}).get('compared_pose_pairs')),
            'ate_rmse': as_float((ate_payload or {}).get('rmse')),
            'ate_mean': as_float((ate_payload or {}).get('mean')),
            'ate_median': as_float((ate_payload or {}).get('median')),
            'ate_std': as_float((ate_payload or {}).get('std')),
            'ate_min': as_float((ate_payload or {}).get('min')),
            'ate_max': as_float((ate_payload or {}).get('max')),
        })

        for split, metrics in [('train', train_metrics), ('test', test_metrics)]:
            for metric in metrics:
                frame_id = metric.get('frame_id', None) if isinstance(metric, dict) else None
                frame_id_str = '' if frame_id is None else str(frame_id)
                frame_rows.append({
                    **base_identity,
                    **test_meta,
                    'run_key': run_key,
                    'analysis_key': analysis_key,
                    'test_variant_rank': variant_rank,
                    'split': split,
                    'frame_id': frame_id_str,
                    'frame_index': pd.to_numeric(frame_id_str, errors='coerce'),
                    'lpips': as_float(metric.get('lpips') if isinstance(metric, dict) else None),
                    'psnr': as_float(metric.get('psnr') if isinstance(metric, dict) else None),
                    'ssim': as_float(metric.get('ssim') if isinstance(metric, dict) else None),
                })

run_summary_df = pd.DataFrame(run_rows)
frame_df = pd.DataFrame(frame_rows)

if run_summary_df.empty:
    run_summary_df = pd.DataFrame(columns=[
        'dataset', 'run_name', 'experiment_name', 'run_dir', 'run_mtime',
        'scene_id', 'ckpt_from_name',
        'sample_rate_from_name', 'n_views_from_name', 'submap_every_from_name', 'profile_from_name', 'parse_pattern',
        'test_metrics_file', 'test_output_label', 'test_protocol', 'use_legacy_test_output',
        'run_key', 'analysis_key',
        'train_json_status', 'test_json_status', 'ate_json_status',
        'train_avg_lpips', 'train_avg_psnr', 'train_avg_ssim',
        'test_avg_lpips', 'test_avg_psnr', 'test_avg_ssim',
        'n_train_frames', 'n_test_frames', 'test_variant_rank',
        'ate_compared_pose_pairs', 'ate_rmse', 'ate_mean', 'ate_median',
        'ate_std', 'ate_min', 'ate_max',
    ])

if frame_df.empty:
    frame_df = pd.DataFrame(columns=[
        'dataset', 'run_name', 'experiment_name', 'run_dir', 'run_mtime',
        'scene_id', 'ckpt_from_name',
        'sample_rate_from_name', 'n_views_from_name', 'submap_every_from_name', 'profile_from_name', 'parse_pattern',
        'test_metrics_file', 'test_output_label', 'test_protocol', 'use_legacy_test_output',
        'run_key', 'analysis_key', 'test_variant_rank',
        'split', 'frame_id', 'frame_index', 'lpips', 'psnr', 'ssim',
    ])

print('run_summary_df shape =', run_summary_df.shape)
print('frame_df shape =', frame_df.shape)
run_summary_df[['dataset', 'run_name', 'test_output_label', 'test_protocol', 'test_avg_psnr']].head()

run_summary_df shape = (9, 37)
frame_df shape = (149, 25)


,dataset,run_name,test_output_label,test_protocol,test_avg_psnr
0,405841,reggs_405841_scene_A_sr10_nv4,sampled,sampled_test,19.048077
1,405841,reggs_405841_scene_C_dl3dv-ckpt_sr10_nv12,sampled,sampled_test,13.001392
2,405841,reggs_405841_scene_C_dl3dv-ckpt_sr30_nv12_sm2_...,sampled,sampled_test,13.552161
3,dl3dv_2,reggs_dl3dv2_dl3dv-ckpt_sr30_nv10,sampled,sampled_test,15.628808
4,dl3dv_2,reggs_dl3dv2_dl3dv-ckpt_sr50_nv10_sm2_stable_v1,sampled,sampled_test,14.574976


## 3. Add generalization-gap fields and ATE comparison notes

In [4]:
for target_df in [run_summary_df]:
    if not target_df.empty:
        target_df['psnr_drop_train_to_test'] = target_df['train_avg_psnr'] - target_df['test_avg_psnr']
        target_df['ssim_drop_train_to_test'] = target_df['train_avg_ssim'] - target_df['test_avg_ssim']
        target_df['lpips_increase_train_to_test'] = target_df['test_avg_lpips'] - target_df['train_avg_lpips']

        high_risk = (
            (target_df['psnr_drop_train_to_test'] > 10.0)
            | (target_df['ssim_drop_train_to_test'] > 0.10)
            | (target_df['lpips_increase_train_to_test'] > 0.10)
        )
        target_df['generalization_risk'] = np.where(high_risk, 'high_gap', 'low_or_unknown')

        target_df['ate_compare_scope'] = 'within_dataset_only'
        target_df['ate_compare_note'] = (
            'ATE scale can differ by dataset. Compare ATE RMSE only within the same dataset.'
        )

if run_summary_df.empty:
    run_summary_best_df = run_summary_df.copy()
else:
    run_summary_best_df = (
        run_summary_df
        .sort_values(['run_key', 'test_variant_rank', 'run_mtime'], ascending=[True, True, False])
        .drop_duplicates(subset=['run_key'], keep='first')
        .copy()
    )

if run_summary_best_df.empty:
    dataset_summary_df = pd.DataFrame(columns=[
        'dataset', 'num_runs',
        'mean_train_avg_psnr', 'mean_test_avg_psnr', 'mean_psnr_drop_train_to_test',
        'mean_train_avg_ssim', 'mean_test_avg_ssim', 'mean_ssim_drop_train_to_test',
        'mean_train_avg_lpips', 'mean_test_avg_lpips', 'mean_lpips_increase_train_to_test',
        'mean_ate_rmse', 'mean_ate_mean',
        'total_train_frames', 'total_test_frames',
        'ate_cross_dataset_comparison',
    ])
else:
    dataset_summary_df = (
        run_summary_best_df.groupby('dataset', dropna=False)
        .agg(
            num_runs=('run_key', 'nunique'),
            mean_train_avg_psnr=('train_avg_psnr', 'mean'),
            mean_test_avg_psnr=('test_avg_psnr', 'mean'),
            mean_psnr_drop_train_to_test=('psnr_drop_train_to_test', 'mean'),
            mean_train_avg_ssim=('train_avg_ssim', 'mean'),
            mean_test_avg_ssim=('test_avg_ssim', 'mean'),
            mean_ssim_drop_train_to_test=('ssim_drop_train_to_test', 'mean'),
            mean_train_avg_lpips=('train_avg_lpips', 'mean'),
            mean_test_avg_lpips=('test_avg_lpips', 'mean'),
            mean_lpips_increase_train_to_test=('lpips_increase_train_to_test', 'mean'),
            mean_ate_rmse=('ate_rmse', 'mean'),
            mean_ate_mean=('ate_mean', 'mean'),
            total_train_frames=('n_train_frames', 'sum'),
            total_test_frames=('n_test_frames', 'sum'),
        )
        .reset_index()
    )
    dataset_summary_df['ate_cross_dataset_comparison'] = 'not_recommended'

ate_summary_cols = [
    'dataset', 'run_name', 'run_key', 'analysis_key',
    'test_output_label', 'test_protocol', 'test_metrics_file', 'test_variant_rank',
    'ate_compared_pose_pairs', 'ate_rmse', 'ate_mean', 'ate_median',
    'ate_std', 'ate_min', 'ate_max', 'ate_compare_scope', 'ate_compare_note',
]
for col in ate_summary_cols:
    if col not in run_summary_df.columns:
        run_summary_df[col] = np.nan
ate_summary_df = run_summary_df[ate_summary_cols].copy()

print('run_summary_best_df shape =', run_summary_best_df.shape)
print('dataset_summary_df shape =', dataset_summary_df.shape)
run_summary_best_df[['dataset', 'run_name', 'test_output_label', 'test_avg_psnr', 'generalization_risk']].head()

run_summary_best_df shape = (9, 43)
dataset_summary_df shape = (3, 16)


,dataset,run_name,test_output_label,test_avg_psnr,generalization_risk
0,405841,reggs_405841_scene_A_sr10_nv4,sampled,19.048077,high_gap
1,405841,reggs_405841_scene_C_dl3dv-ckpt_sr10_nv12,sampled,13.001392,high_gap
2,405841,reggs_405841_scene_C_dl3dv-ckpt_sr30_nv12_sm2_...,sampled,13.552161,high_gap
3,dl3dv_2,reggs_dl3dv2_dl3dv-ckpt_sr30_nv10,sampled,15.628808,high_gap
4,dl3dv_2,reggs_dl3dv2_dl3dv-ckpt_sr50_nv10_sm2_stable_v1,sampled,14.574976,high_gap


## 4. Export csv summaries into results/part2

In [5]:
run_summary_csv = FINAL_DIR / 'part2_run_summary.csv'
run_summary_all_variants_csv = FINAL_DIR / 'part2_run_summary_all_test_variants.csv'
ate_summary_csv = FINAL_DIR / 'part2_ate_summary.csv'
dataset_summary_csv = FINAL_DIR / 'part2_dataset_summary.csv'
frame_metrics_csv = FRAME_DIR / 'part2_frame_metrics.csv'
run_inventory_csv = QC_DIR / 'part2_run_inventory.csv'
skipped_runs_csv = QC_DIR / 'part2_skipped_runs.csv'

# Reorder all-variants table:
# 1) dataset, run_name first
# 2) key metrics next (result-focused)
# 3) run parameters in the middle
# 4) paths/status/protocol metadata at the end
front_cols = [
    'dataset', 'run_name',
]

result_cols = [
    'ate_rmse',
    'train_avg_psnr', 'test_avg_psnr', 'psnr_drop_train_to_test',
    'ate_mean', 'ate_median', 'ate_std', 'ate_min', 'ate_max', 'ate_compared_pose_pairs',
    'train_avg_ssim', 'test_avg_ssim', 'ssim_drop_train_to_test',
    'train_avg_lpips', 'test_avg_lpips', 'lpips_increase_train_to_test',
    'n_train_frames', 'n_test_frames',
    'generalization_risk',
]

param_cols = [
    'scene_id',
    'ckpt_from_name',
    'sample_rate_from_name', 'n_views_from_name',
    'submap_every_from_name',
    'profile_from_name',
]

tail_cols = [
    'experiment_name',
    'test_output_label', 'test_protocol', 'test_metrics_file', 'test_variant_rank',
    'use_legacy_test_output',
    'run_dir', 'run_mtime',
    'parse_pattern',
    'run_key', 'analysis_key',
    'train_json_status', 'test_json_status', 'ate_json_status',
    'ate_compare_scope', 'ate_compare_note',
]

preferred_order = front_cols + result_cols + param_cols + tail_cols
ordered_existing = [c for c in preferred_order if c in run_summary_df.columns]
remaining_cols = [c for c in run_summary_df.columns if c not in ordered_existing]
run_summary_export_df = run_summary_df[ordered_existing + remaining_cols].copy()

run_summary_best_df.sort_values(['dataset', 'run_name']).to_csv(run_summary_csv, index=False)
run_summary_export_df.sort_values(['dataset', 'run_name', 'test_variant_rank', 'test_output_label']).to_csv(run_summary_all_variants_csv, index=False)
ate_summary_df.sort_values(['dataset', 'run_name', 'test_variant_rank', 'test_output_label']).to_csv(ate_summary_csv, index=False)
dataset_summary_df.sort_values(['dataset']).to_csv(dataset_summary_csv, index=False)
frame_df.sort_values(['dataset', 'run_name', 'test_variant_rank', 'test_output_label', 'split', 'frame_index', 'frame_id']).to_csv(frame_metrics_csv, index=False)
inventory_df.sort_values(['dataset', 'run_name']).to_csv(run_inventory_csv, index=False)
skipped_runs_df.sort_values(['dataset', 'run_name']).to_csv(skipped_runs_csv, index=False)

print('run_summary_csv (best variant per run) =', run_summary_csv)
print('run_summary_all_variants_csv =', run_summary_all_variants_csv)
print('ate_summary_csv =', ate_summary_csv)
print('dataset_summary_csv =', dataset_summary_csv)
print('frame_metrics_csv =', frame_metrics_csv)
print('run_inventory_csv =', run_inventory_csv)
print('skipped_runs_csv =', skipped_runs_csv)

run_summary_csv (best variant per run) = /home/bzhang512/CV_Project/results/part2/final/part2_run_summary.csv
run_summary_all_variants_csv = /home/bzhang512/CV_Project/results/part2/final/part2_run_summary_all_test_variants.csv
ate_summary_csv = /home/bzhang512/CV_Project/results/part2/final/part2_ate_summary.csv
dataset_summary_csv = /home/bzhang512/CV_Project/results/part2/final/part2_dataset_summary.csv
frame_metrics_csv = /home/bzhang512/CV_Project/results/part2/frame/part2_frame_metrics.csv
run_inventory_csv = /home/bzhang512/CV_Project/results/part2/qc/part2_run_inventory.csv
skipped_runs_csv = /home/bzhang512/CV_Project/results/part2/qc/part2_skipped_runs.csv


## 5. Quick checks (coverage, skipped runs, frame counts)

In [6]:
def show_df(df, n=10):
    if df.empty:
        print('empty dataframe')
        return
    try:
        display(df.head(n))
    except NameError:
        print(df.head(n).to_string(index=False))

print('candidate runs =', len(inventory_df))
print('complete runs =', len(complete_runs_df))
print('skipped runs =', len(skipped_runs_df))
print('run_summary rows (all test variants) =', len(run_summary_df))
print('run_summary rows (best per run) =', len(run_summary_best_df))
print('frame rows =', len(frame_df))

print('\nRun summary (best variant per run):')
cols = [
    'dataset', 'scene_id', 'run_name', 'ckpt_from_name', 'profile_from_name',
    'test_output_label', 'test_protocol',
    'train_avg_psnr', 'test_avg_psnr', 'psnr_drop_train_to_test',
    'train_avg_ssim', 'test_avg_ssim', 'ssim_drop_train_to_test',
    'train_avg_lpips', 'test_avg_lpips', 'lpips_increase_train_to_test',
    'n_train_frames', 'n_test_frames',
    'ate_rmse', 'generalization_risk',
    'sample_rate_from_name', 'n_views_from_name', 'submap_every_from_name',
    'parse_pattern',
    'test_variant_rank',
]
cols = [c for c in cols if c in run_summary_best_df.columns]
show_df(run_summary_best_df[cols].sort_values(['dataset', 'scene_id', 'run_name']), n=50)

print('\nAll test variants (sample rows):')
variant_cols = [
    'dataset', 'scene_id', 'run_name',
    'test_metrics_file', 'test_output_label', 'test_protocol', 'test_variant_rank',
    'test_avg_psnr', 'test_avg_ssim', 'test_avg_lpips',
    'n_test_frames',
]
variant_cols = [c for c in variant_cols if c in run_summary_df.columns]
show_df(run_summary_df[variant_cols].sort_values(['dataset', 'run_name', 'test_variant_rank']), n=80)

print('\nSkipped runs and reasons:')
show_df(skipped_runs_df[['dataset', 'run_name', 'missing_files', 'skip_reason']].sort_values(['dataset', 'run_name']), n=50)

if not frame_df.empty:
    frame_count_df = (
        frame_df.groupby(['dataset', 'run_name', 'test_output_label', 'split'], dropna=False)
        .size()
        .reset_index(name='num_frames')
        .sort_values(['dataset', 'run_name', 'test_output_label', 'split'])
    )
    print('\nFrame counts per run/test_variant/split:')
    show_df(frame_count_df, n=100)

print('\nATE note: compare ATE RMSE only within the same dataset; avoid direct cross-dataset ranking.')

candidate runs = 10
complete runs = 9
skipped runs = 1
run_summary rows (all test variants) = 9
run_summary rows (best per run) = 9
frame rows = 149

Run summary (best variant per run):


,dataset,scene_id,run_name,ckpt_from_name,profile_from_name,test_output_label,test_protocol,train_avg_psnr,test_avg_psnr,psnr_drop_train_to_test,...,lpips_increase_train_to_test,n_train_frames,n_test_frames,ate_rmse,generalization_risk,sample_rate_from_name,n_views_from_name,submap_every_from_name,parse_pattern,test_variant_rank
0,405841,A,reggs_405841_scene_A_sr10_nv4,NaN,NaN,sampled,sampled_test,49.940655,19.048077,30.892578,...,0.508322,4,2,0.661653,high_gap,10,4,NaN,405841_old,1
1,405841,C,reggs_405841_scene_C_dl3dv-ckpt_sr10_nv12,dl3dv,NaN,sampled,sampled_test,45.948605,13.001392,32.947212,...,0.554433,12,13,18.466274,high_gap,10,12,NaN,fallback,1
2,405841,C,reggs_405841_scene_C_dl3dv-ckpt_sr30_nv12_sm2_...,dl3dv,stable_v1,sampled,sampled_test,47.408217,13.552161,33.856057,...,0.548181,12,4,17.423334,high_gap,30,12,2.0,405841_new,1
3,dl3dv_2,main,reggs_dl3dv2_dl3dv-ckpt_sr30_nv10,dl3dv,NaN,sampled,sampled_test,51.774280,15.628808,36.145473,...,0.478668,10,10,2.377937,high_gap,30,10,NaN,fallback,1
4,dl3dv_2,main,reggs_dl3dv2_dl3dv-ckpt_sr50_nv10_sm2_stable_v1,dl3dv,stable_v1,sampled,sampled_test,50.933286,14.574976,36.358309,...,0.524957,10,6,2.308413,high_gap,50,10,2.0,general_new,1
5,dl3dv_2,main,reggs_dl3dv2_dl3dv-ckpt_sr50_nv10_sm3_stable_v...,dl3dv,stable_v2_overfit,sampled,sampled_test,51.784956,14.628931,37.156025,...,0.524187,10,6,2.448270,high_gap,50,10,3.0,general_new,1
6,dl3dv_2,main,reggs_dl3dv2_re10k-ckpt_sr30_nv8,re10k,NaN,sampled,sampled_test,52.825459,15.689012,37.136448,...,0.465091,8,10,2.325984,high_gap,30,8,NaN,fallback,1
7,re10k_1,main,reggs_re10k1_re10k-ckpt_sr50_nv9_sm2_compariso...,re10k,comparison_check,sampled,sampled_test,41.368002,25.861177,15.506825,...,0.119452,9,6,0.008739,high_gap,50,9,2.0,general_new,1
8,re10k_1,main,reggs_re10k1_sr30_nv8,NaN,NaN,sampled,sampled_test,42.275033,25.001969,17.273063,...,0.139360,8,9,0.010643,high_gap,30,8,NaN,general_old,1



All test variants (sample rows):


,dataset,scene_id,run_name,test_metrics_file,test_output_label,test_protocol,test_variant_rank,test_avg_psnr,test_avg_ssim,test_avg_lpips,n_test_frames
0,405841,A,reggs_405841_scene_A_sr10_nv4,eval_test.json,sampled,sampled_test,1,19.048077,0.561145,0.587988,2
1,405841,C,reggs_405841_scene_C_dl3dv-ckpt_sr10_nv12,eval_test.json,sampled,sampled_test,1,13.001392,0.329157,0.634676,13
2,405841,C,reggs_405841_scene_C_dl3dv-ckpt_sr30_nv12_sm2_...,eval_test.json,sampled,sampled_test,1,13.552161,0.372611,0.610124,4
3,dl3dv_2,main,reggs_dl3dv2_dl3dv-ckpt_sr30_nv10,eval_test.json,sampled,sampled_test,1,15.628808,0.442496,0.480798,10
4,dl3dv_2,main,reggs_dl3dv2_dl3dv-ckpt_sr50_nv10_sm2_stable_v1,eval_test.json,sampled,sampled_test,1,14.574976,0.403971,0.527627,6
5,dl3dv_2,main,reggs_dl3dv2_dl3dv-ckpt_sr50_nv10_sm3_stable_v...,eval_test.json,sampled,sampled_test,1,14.628931,0.412569,0.525824,6
6,dl3dv_2,main,reggs_dl3dv2_re10k-ckpt_sr30_nv8,eval_test.json,sampled,sampled_test,1,15.689012,0.446930,0.466456,10
7,re10k_1,main,reggs_re10k1_re10k-ckpt_sr50_nv9_sm2_compariso...,eval_test.json,sampled,sampled_test,1,25.861177,0.897456,0.124374,6
8,re10k_1,main,reggs_re10k1_sr30_nv8,eval_test.json,sampled,sampled_test,1,25.001969,0.879175,0.143630,9



Skipped runs and reasons:


,dataset,run_name,missing_files,skip_reason
3,405841,reggs_405841_scene_full_dl3dv-ckpt_sr30_nv20_s...,eval_train.json;ate_aligned.json;eval_test*.json,missing:eval_train.json;ate_aligned.json;eval_...



Frame counts per run/test_variant/split:


,dataset,run_name,test_output_label,split,num_frames
0,405841,reggs_405841_scene_A_sr10_nv4,sampled,test,2
1,405841,reggs_405841_scene_A_sr10_nv4,sampled,train,4
2,405841,reggs_405841_scene_C_dl3dv-ckpt_sr10_nv12,sampled,test,13
3,405841,reggs_405841_scene_C_dl3dv-ckpt_sr10_nv12,sampled,train,12
4,405841,reggs_405841_scene_C_dl3dv-ckpt_sr30_nv12_sm2_...,sampled,test,4
5,405841,reggs_405841_scene_C_dl3dv-ckpt_sr30_nv12_sm2_...,sampled,train,12
6,dl3dv_2,reggs_dl3dv2_dl3dv-ckpt_sr30_nv10,sampled,test,10
7,dl3dv_2,reggs_dl3dv2_dl3dv-ckpt_sr30_nv10,sampled,train,10
8,dl3dv_2,reggs_dl3dv2_dl3dv-ckpt_sr50_nv10_sm2_stable_v1,sampled,test,6
9,dl3dv_2,reggs_dl3dv2_dl3dv-ckpt_sr50_nv10_sm2_stable_v1,sampled,train,10



ATE note: compare ATE RMSE only within the same dataset; avoid direct cross-dataset ranking.
